In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pl.Config.set_tbl_rows(100)

DATA_PATH = r"E:\Databard\DLD\dld-clean.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Rows: {df.height:,} | Columns: {df.width}")
print(df.schema)

Rows: 566,235 | Columns: 19
Schema({'instance_date': Date, 'area_name_en': String, 'building_name_en': String, 'project_name_en': String, 'master_project_en': String, 'nearest_landmark_en': String, 'nearest_metro_en': String, 'nearest_mall_en': String, 'has_parking': Int64, 'procedure_area': Float64, 'actual_worth': Float64, 'meter_sale_price': Float64, 'year': Int32, 'month': Int8, 'sale_type': String, 'property_cat': String, 'district': String, 'neighborhood': String, 'bedrooms': Int8})


## Price Heatmap by Neighborhood Over Time

### What are we looking at?
A heatmap where each row is a neighborhood, each column is a year, and the colour represents the **median price per sqm** — the darker (or warmer) the cell, the more expensive that area was in that year.

### Why this chart?
Bar charts and line charts show one dimension well. This heatmap shows **133 neighborhoods across 5 years simultaneously** — something no other chart type can do cleanly. It lets the eye immediately spot patterns that would take dozens of individual charts to find.

### What can we learn from it?
- **Which areas are consistently expensive** — rows that are uniformly dark across all years
- **Which areas heated up fast** — rows that transition from light to dark left-to-right, revealing the fastest-appreciating markets
- **Market-wide trends** — if the entire heatmap gets darker over time, prices rose broadly across Dubai, not just in isolated pockets
- **Hidden gems** — areas that are still relatively light (cheap) despite neighbours darkening fast, potentially signalling lagging appreciation or upcoming opportunity
- **The two-speed market** — whether premium areas (Marina, Downtown) are pulling further ahead of mid-market areas, or whether the gap is closing

In [2]:
heatmap_data = (
    df.group_by(["neighborhood", "year"])
      .agg(pl.median("meter_sale_price").alias("median_sqm"))
      .sort(["neighborhood", "year"])
)

# Pivot: rows = neighborhood, columns = year
heatmap_pivot = heatmap_data.pivot(
    on="year",
    index="neighborhood",
    values="median_sqm",
    aggregate_function="first",
).sort("neighborhood")

years = sorted(df["year"].unique().to_list())
neighborhoods = heatmap_pivot["neighborhood"].to_list()
z_values = heatmap_pivot.select(
    [pl.col(str(y)) for y in years]
).to_numpy()

fig = go.Figure(data=go.Heatmap(
    z=z_values,
    x=[str(y) for y in years],
    y=neighborhoods,
    colorscale="YlOrRd",
    colorbar=dict(title="AED / sqm"),
    hoverongaps=False,
))

fig.update_layout(
    title="Median Price per sqm by Neighborhood & Year",
    xaxis_title="Year",
    yaxis_title="Neighborhood",
    height=1400,
    yaxis=dict(tickfont=dict(size=10)),
)

fig.show()

### What the heatmap tells us

**The market heated up broadly, not selectively.** Across almost every neighborhood, cells darken noticeably from 2021 to 2025. This was not a story of one or two hot pockets pulling ahead, but a market-wide appreciation cycle driven by post-pandemic demand, population growth, and a wave of global capital inflows into Dubai real estate.

**"Island" stands out as the persistent price outlier.** The single darkest row throughout the entire chart belongs to "Island", which in our dataset traces back to the DLD area code `Island 2`, widely understood to be **The World Islands**, a collection of ~300 private man-made islands off the Dubai coastline. With only 267 transactions over 5 years (vs tens of thousands for mainstream areas), this is an ultra-exclusive, ultra-illiquid market. The extreme price per sqm reflects scarcity and prestige, but the low volume means each individual transaction moves the median significantly. Treat this row as directionally correct, not statistically robust.

**Premium corridors are clearly visible.** Downtown Dubai, Dubai Marina, Palm Jumeirah, and Dubai Creek Harbour form a consistently warm band. They were expensive in 2021 and got more expensive each year. These are mature, branded destinations with limited new supply, so price appreciation here reflects genuine demand pressure rather than new development activity.

**The fastest heat-up story belongs to mid-market areas.** Neighborhoods like Jumeirah Village Circle, Arjan, and Dubai South show a more dramatic colour shift than the premium tier, starting relatively light in 2021 and darkening sharply by 2023-2024. This reflects the off-plan boom: mass-market investors chasing yield flooded these areas with pre-registration purchases, compressing supply and lifting prices faster than the established luxury tier.

**Grey cells (missing data) are informative too.** A handful of neighborhoods show gaps in certain years, meaning not enough transactions occurred that year to produce a reliable median. These are typically very small or very new communities. Gaps are not errors; they are a signal of illiquidity.

## Off-Plan vs Ready Market Share Over Time

### What are we looking at?
A stacked area chart where each band represents the monthly transaction volume for Off-Plan and Ready sales respectively. The relative thickness of each band shows how the market composition has shifted year by year.

### Why this chart?
The Off-Plan vs Ready split is one of the most structurally important signals in Dubai real estate. It tells us whether buyers are betting on future delivery or transacting on existing stock, which reflects confidence levels, developer activity, and the overall market cycle phase.

### What can we learn from it?
- **How dominant off-plan has become** and whether that dominance is growing, stabilising, or reversing
- **Market confidence cycles** - off-plan surges when buyers trust future delivery; ready market surges when buyers want certainty
- **Developer pipeline pressure** - a sustained off-plan surge means a large supply wave is coming in 3-5 years, which matters for future pricing
- **Timing of market shifts** - pinpoint exactly when the balance tipped and what external events may have triggered it

In [5]:
split_trend = (
    df.group_by(["year", "month", "sale_type"])
      .agg(pl.len().alias("transactions"))
      .sort(["year", "month", "sale_type"])
      .with_columns(
          pl.date(pl.col("year"), pl.col("month"), pl.lit(1)).alias("date")
      )
)

fig = px.area(
    split_trend,
    x="date", y="transactions", color="sale_type",
    title="Off-Plan vs Ready: Monthly Transaction Volume Over Time",
    labels={"transactions": "Transactions", "date": "", "sale_type": "Type"},
    color_discrete_map={"Off-Plan": "#EF553B", "Ready": "#636EFA"},
)
fig.update_layout(hovermode="x unified")
fig.show()

### What the chart tells us

**Off-plan started at parity and never looked back.** In early 2021, off-plan and ready transactions were roughly equal at around 50% each. From that point the chart tells a single, unbroken story: off-plan's share climbed steadily, reaching 75-80% by 2025 and pushing toward 80%+ into 2026. There was no reversal, no correction, no moment where the ready market fought back meaningfully.

**The pace of the shift is the real story.** This was not an overnight flip but a grinding, consistent re-weighting of the market over four years. Each year, off-plan took a few more percentage points of share. That kind of steady trend is more structurally significant than a sharp spike, because it suggests a fundamental change in buyer behaviour and developer strategy rather than a temporary sentiment shift.

**What drove it.** The 2022-2023 period was the acceleration phase, coinciding with Dubai's post-pandemic population boom, the expansion of the Golden Visa programme, and a wave of international capital seeking yield in a tax-free environment. Developers responded by launching more projects with attractive payment plans, which made off-plan accessible to a far wider pool of buyers than ever before.

**The pipeline risk is now very real.** By 2025, roughly 75-80 out of every 100 residential transactions in Dubai were off-plan. Assuming average delivery timelines of 2 to 3 years, this means an exceptionally large wave of completed units is due between 2025 and 2028. Whether the market can absorb that supply without a meaningful price correction depends entirely on whether the demand side continues to grow at the same pace as the supply side. This chart does not answer that question, but it defines why the question matters.

**Ready market held its absolute volume.** The shrinking blue band does not mean fewer ready transactions in absolute terms. Total market volume expanded significantly over the period, so the ready market simply grew more slowly than off-plan. For end-users and secondary market investors, the ready segment remained active throughout.

## Bedroom Mix by Neighborhood

### What are we looking at?
A 100% stacked horizontal bar chart where each bar is a neighborhood. The coloured segments within each bar show the proportion of transactions by bedroom count, from Studio to 4+ bedrooms. Every bar adds up to 100%, so we are comparing composition, not volume.

### Why this chart?
Transaction volume and price alone do not tell you what kind of city a neighborhood is. Two areas can have identical median prices but serve completely different buyers. This chart reveals the product mix: which areas are dominated by studio investors, which are family-oriented 3-bedroom communities, and which offer a broad spread across all sizes.

### What can we learn from it?
- **Investor vs end-user signal** - studio and 1-bedroom heavy areas attract short-term investors and renters; 3-bedroom and above skew toward families and long-term residents
- **Developer strategy fingerprints** - off-plan dominated areas often show a very specific bedroom mix that reflects what one or two major developers chose to build
- **Diversification of supply** - areas with an even spread across bedroom types tend to have more resilient demand, as they serve multiple buyer segments simultaneously
- **Mismatch opportunities** - if a premium area is surprisingly studio-heavy, it may signal an oversupply of small units relative to what the neighborhood's price point would nor

In [ ]:
def bedroom_mix_chart(property_cat, top_n=20):
    top_neighborhoods = (
        df.filter(pl.col("property_cat") == property_cat)
          .group_by("neighborhood")
          .agg(pl.len().alias("transactions"))
          .sort("transactions", descending=True)
          .head(top_n)
          ["neighborhood"].to_list()
    )

    mix = (
        df.filter(pl.col("property_cat") == property_cat)
          .filter(pl.col("neighborhood").is_in(top_neighborhoods))
          .filter(pl.col("bedrooms") <= 4)
          .with_columns(
              pl.when(pl.col("bedrooms") == 0).then(pl.lit("Studio"))
                .when(pl.col("bedrooms") == 1).then(pl.lit("1 B/R"))
                .when(pl.col("bedrooms") == 2).then(pl.lit("2 B/R"))
                .when(pl.col("bedrooms") == 3).then(pl.lit("3 B/R"))
                .otherwise(pl.lit("4+ B/R"))
                .alias("bedroom_label")
          )
          .group_by(["neighborhood", "bedroom_label"])
          .agg(pl.len().alias("count"))
    )

    totals = mix.group_by("neighborhood").agg(pl.col("count").sum().alias("total"))
    mix = (
        mix.join(totals, on="neighborhood")
           .with_columns((pl.col("count") / pl.col("total") * 100).round(1).alias("pct"))
           .sort(["neighborhood", "bedroom_label"])
    )

    neighborhood_order = (
        df.filter(pl.col("property_cat") == property_cat)
          .filter(pl.col("neighborhood").is_in(top_neighborhoods))
          .group_by("neighborhood")
          .agg(pl.len().alias("total"))
          .sort("total", descending=False)
          ["neighborhood"].to_list()
    )

    BEDROOM_COLORS = {
        "Studio":  "#636EFA",
        "1 B/R":   "#00CC96",
        "2 B/R":   "#FFA15A",
        "3 B/R":   "#EF553B",
        "4+ B/R":  "#AB63FA",
    }

    fig = px.bar(
        mix,
        x="pct", y="neighborhood",
        color="bedroom_label",
        orientation="h",
        title=f"Bedroom Mix — {property_cat}s (Top {top_n} by Volume)",
        labels={"pct": "Share (%)", "neighborhood": "", "bedroom_label": "Bedrooms"},
        category_orders={
            "neighborhood": neighborhood_order,
            "bedroom_label": ["Studio", "1 B/R", "2 B/R", "3 B/R", "4+ B/R"],
        },
        color_discrete_map=BEDROOM_COLORS,
    )
    fig.update_layout(
        height=700,
        xaxis=dict(ticksuffix="%"),
        barmode="stack",
        legend=dict(traceorder="normal"),
        hovermode="y unified",
    )
    fig.show()

bedroom_mix_chart("Flat")
bedroom_mix_chart("Villa")

### What the charts tell us

**Flats: 1 B/R dominates almost everywhere, but the studio story is the interesting one.** Across nearly every flat market, 1-bedroom units make up the largest single segment, reflecting Dubai's deep pool of single professionals and small investor units. But the studio share varies enormously and that variation is the real signal. Dubai Maritime City, Jumeirah Lake Towers, and Downtown Dubai all show a meaningful studio band, reflecting mature investor markets where small units offer accessible price points and rental yield. Meydan stands out as almost entirely 1 B/R and 2 B/R with virtually no studios, suggesting a more end-user oriented buyer profile despite being a relatively new community.

**JVC's flat mix is surprisingly balanced.** For the highest-volume area on the entire chart, Jumeirah Village Circle shows a healthy spread across Studio, 1 B/R, and 2 B/R. This is not a single-product market. Developers built a diverse range of unit sizes, which is one reason JVC attracted such broad demand and sustained its volume year after year.

**Dubai Creek Harbour skews large.** The flat chart shows DCH with a notably high share of 2 B/R and 3 B/R compared to peers at similar price points. This reflects Emaar's product strategy there: fewer studios, more family-sized units positioned at the premium end of the flat market.

**Villas: the market splits cleanly into two camps.** On the villa chart, every single bar is orange (3 B/R) and purple (4+ B/R) with almost no smaller units. This is expected. But the split between 3 B/R and 4+ B/R varies significantly by area and reveals developer intent.

**Villanova and Dubai South are 3 B/R and 4 B/R townhouse communities.** Their bars are almost entirely orange and red, with very little 4+ B/R purple. These are affordable family villa communities where the product is compact townhouses rather than large standalone villas.

**Al Barari and Dubai Investment Park skew heavily to 4+ B/R.** The purple band dominates these areas, confirming their positioning as large-format, high-end villa destinations catering to wealthy families rather than first-time villa buyers.

**Arabian Ranches 2 is the only area showing a green (1 B/R) segment.** A small but visible green band appears, which likely represents studio or 1-bedroom ancillary units or staff quarters registered separately. It is a data curiosity rather than a meaningful market signal.

**Mohammed Bin Rashid City villas show a balanced 3 B/R and 4+ B/R split.** This reflects MBR City's role as a mixed master community containing both mid-size townhouses and large luxury villas under the same district umbrella, serving multiple price points simultaneously.

## Price Distribution by Bedroom Count

### What are we looking at?
A box plot where each box represents one bedroom category. The box shows the interquartile range (25th to 75th percentile), the line inside is the median, and the whiskers extend to capture the broader distribution. Points beyond the whiskers are outliers.

### Why this chart?
Averages and medians hide the shape of a distribution. Two bedroom categories can have the same median price but wildly different spreads. A studio in International City and a studio in Downtown Dubai are both studios, but they exist in completely different price universes. Box plots make that spread visible at a glance and expose outliers that averages simply wash out.

### What can we learn from it?
- **How wide the price range is within each bedroom type** - a tall box means high variance, meaning the same bedroom count can mean very different things depending on location and quality
- **Where the outliers live** - dots above the whisker represent ultra-luxury transactions that sit far above the typical market for that bedroom count
- **Whether larger bedrooms command proportionally higher prices** - or whether the relationship between size and price is non-linear
- **Which bedroom segment has the most pricing uncertainty** - relevant for buyers and investors trying to benchmark a specific deal against the market

In [ ]:
box_data = (
    df.filter(pl.col("bedrooms") <= 5)
      .filter(pl.col("year") == 2025)
      .with_columns(
          pl.when(pl.col("bedrooms") == 0).then(pl.lit("Studio"))
            .when(pl.col("bedrooms") == 1).then(pl.lit("1 B/R"))
            .when(pl.col("bedrooms") == 2).then(pl.lit("2 B/R"))
            .when(pl.col("bedrooms") == 3).then(pl.lit("3 B/R"))
            .when(pl.col("bedrooms") == 4).then(pl.lit("4 B/R"))
            .otherwise(pl.lit("5 B/R"))
            .alias("bedroom_label")
      )
)

LABEL_ORDER = ["Studio", "1 B/R", "2 B/R", "3 B/R", "4 B/R", "5 B/R"]

fig = px.box(
    box_data,
    x="bedroom_label",
    y="actual_worth",
    color="property_cat",
    title="Sale Price Distribution by Bedroom Count — 2025",
    labels={
        "actual_worth":    "Sale Price (AED)",
        "bedroom_label":   "Bedrooms",
        "property_cat":    "Type",
    },
    category_orders={"bedroom_label": LABEL_ORDER},
    color_discrete_map={"Flat": "#636EFA", "Villa": "#EF553B"},
    points=False,
)

fig.update_layout(
    height=600,
    yaxis=dict(tickformat=",.0f"),
    hovermode="x unified",
    boxmode="group",
)
fig.show()

### What the chart tells us

**Outliers still dominate the axis, but the picture is cleaner in 2025.** Restricting to 2025 data reduces the Y axis to 250 million AED, which is still extreme but allows the smaller bedroom categories to become more readable. The culprit is again the 5 B/R villa segment, where the upper whisker stretches to 250 million AED, and the 5 B/R flat segment reaching around 175 million AED. These are super-prime trophy transactions that represent a tiny fraction of deals but define the ceiling of what Dubai's residential market can produce.

**Studios and 1 B/R flats are tightly priced in 2025.** The boxes for these two categories are compact and sit close to zero on this scale, reflecting their nature as the most standardised and liquid segment of the market. Most studio and 1 B/R flat transactions in 2025 completed in a relatively narrow band, giving buyers reasonable price discovery and low variance.

**2 B/R and 3 B/R flats show growing spread.** As bedroom count rises, the interquartile range widens noticeably. A 2 B/R flat whisker reaches around 50 million AED and a 3 B/R reaches around 80 million AED. This reflects the increasing role of location and finish quality at larger sizes. Two 2-bedroom flats in JVC and Downtown Dubai can differ by an order of magnitude in price.

**Villas at 4 B/R and below are more tightly distributed than flats.** The red villa boxes for 2 B/R through 4 B/R are compact relative to their flat counterparts. This reflects the more homogeneous nature of villa communities. Most villa buyers in 2025 are transacting within master-planned developments with standardised product, which anchors pricing more than the heterogeneous flat market.

**5 B/R villas are the single most volatile segment.** The red whisker for 5 B/R villas extends to 250 million AED while the box itself sits barely above zero. The interquartile range is almost invisible at this scale, meaning half of all 5 B/R villa transactions in 2025 clustered tightly at a relatively accessible price point, while the rest spread across an enormous range driven by ultra-luxury Palm and MBR City mansions.

## Rolling 3-Month Transaction Volume

### What are we looking at?
A line chart showing the total number of residential transactions per month, smoothed using a 3-month rolling average. The raw monthly count is shown as a faint background line, while the smoothed trend sits on top.

### Why this chart?
Raw monthly transaction counts are noisy. Public holidays, Ramadan timing, and reporting lags create month-to-month swings that obscure the real trend. A 3-month rolling average removes that noise without over-smoothing the signal, making it much easier to identify genuine market acceleration, deceleration, and turning points.

### What can we learn from it?
- **Whether the market is in an uptrend, downtrend, or plateau** at any given point in time
- **Turning points** where momentum shifted, which often correspond to external events like interest rate changes, visa policy updates, or global macro shocks
- **Seasonality patterns** that repeat annually, confirming structural calendar effects versus one-off disruptions
- **The pace of growth** by comparing the slope of the trend line across different periods

In [ ]:
monthly = (
    df.group_by(["year", "month"])
      .agg(pl.len().alias("transactions"))
      .sort(["year", "month"])
      .with_columns(
          pl.date(pl.col("year"), pl.col("month"), pl.lit(1)).alias("date")
      )
      .with_columns(
          pl.col("transactions")
            .rolling_mean(window_size=3, min_periods=1)
            .alias("rolling_3m")
      )
)

fig = go.Figure()

# Raw monthly — faint background
fig.add_trace(go.Scatter(
    x=monthly["date"].to_list(),
    y=monthly["transactions"].to_list(),
    mode="lines",
    name="Monthly (raw)",
    line=dict(color="lightsteelblue", width=1),
    opacity=0.5,
))

# Rolling 3-month average
fig.add_trace(go.Scatter(
    x=monthly["date"].to_list(),
    y=monthly["rolling_3m"].to_list(),
    mode="lines",
    name="3-Month Rolling Avg",
    line=dict(color="#EF553B", width=2.5),
))

fig.update_layout(
    title="Residential Transaction Volume: Rolling 3-Month Average",
    xaxis_title="",
    yaxis_title="Transactions",
    height=500,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

### What the chart tells us

**The trend is almost perfectly unbroken from 2021 to 2026.** Starting from around 2,000 transactions per month in early 2021, the rolling average climbed without a meaningful sustained reversal all the way to a peak of around 16,500-17,000 in late 2025. That is roughly an 8x increase in monthly transaction volume over five years. Very few real estate markets in the world can show a chart that looks like this.

**2021 to mid-2022: the post-pandemic ignition.** The initial slope is steep but orderly, rising from 2k to around 4,500 per month. This phase corresponds to the immediate post-lockdown reopening, the introduction of the 10-year Golden Visa, and Dubai's early emergence as a destination for remote workers and global nomads. Demand surged before developer supply could respond.

**2022 to 2024: steady institutionalisation of the market.** The rolling average ticked steadily higher from 4,500 to around 10,000 per month. This was not a frenzy; the slope is consistent and smooth. It reflects a market that had matured from an opportunistic bounce into a structurally growing one, fed by continued population inflow, a maturing off-plan pipeline delivering investor confidence, and a secondary market starting to become more liquid.

**Mid-2024 to early 2025: the first serious wobble.** There is a visible dip in the rolling average around mid-2024 to early 2025, dropping from the 15,000 range back toward 12,000. This is the clearest sign of a market taking a breath, potentially reflecting rate sensitivity, a temporary slowdown in new project launches, or buyer hesitation as prices reached all-time highs. Importantly, the dip is a deceleration, not a reversal.

**2025: recovery and new peak.** The rolling average recovered quickly and pushed to a new high of around 16,500 to 17,000 by late 2025, confirming that the mid-2024 softness was temporary. The market found its footing and resumed the uptrend.

**The 2026 drop-off is a data artefact, not a crash.** The sharp decline visible at the very right edge of the chart reflects incomplete data for the final months rather than an actual collapse in activity. Transactions registered toward the very end of the dataset period have not fully captured, so the trailing months always undercount. Treat the 2026 right edge as noise.

## Year-on-Year Price Growth by Neighborhood

### What are we looking at?
A horizontal bar chart ranking neighborhoods by their cumulative median price per sqm growth from 2021 to 2025. Each bar shows how much more expensive (or cheaper) a neighborhood became over the full four-year period, expressed as a percentage change.

### Why this chart?
Price levels tell you where a market is. Price growth tells you where it has been moving and at what speed. Two neighborhoods at the same price per sqm today may have arrived from very different starting points. One could be a mature market that barely moved; the other could be an emerging area that doubled in four years. That difference matters enormously for investors assessing momentum versus value.

### What can we learn from it?
- **Which neighborhoods delivered the highest appreciation** over the full cycle, not just a single year
- **Where the fastest growth occurred relative to the current price level** — high growth from a low base is a different story to high growth from an already expensive base
- **Whether premium areas outpaced mid-market or vice versa** — rising tides do not always lift all boats equally
- **Which areas are potential laggards** with low growth despite a broadly rising market, which could signal structural oversupply, limited demand, or simply a later entry into the cycle

In [ ]:
EXCLUDE = ["Island", "Trade Center", "Palm Jabal Ali"]

price_by_year = (
    df.filter(pl.col("neighborhood").is_in(EXCLUDE).not_())
      .filter(pl.col("year").is_in([2021, 2025]))
      .group_by(["neighborhood", "year"])
      .agg(pl.median("meter_sale_price").alias("median_sqm"))
)

# Keep only neighborhoods that have data in BOTH 2021 and 2025
both_years = (
    price_by_year.group_by("neighborhood")
                 .agg(pl.len().alias("n"))
                 .filter(pl.col("n") == 2)
                 ["neighborhood"].to_list()
)

yoy = (
    price_by_year.filter(pl.col("neighborhood").is_in(both_years))
    .pivot(on="year", index="neighborhood", values="median_sqm", aggregate_function="first")
    .rename({"2021": "price_2021", "2025": "price_2025"})
    .with_columns(
        ((pl.col("price_2025") - pl.col("price_2021")) / pl.col("price_2021") * 100)
        .round(1)
        .alias("growth_pct")
    )
    .sort("growth_pct", descending=True)
)

# Top 30 and bottom 10 for context
top = yoy.head(30)
bottom = yoy.tail(10)
chart_data = pl.concat([top, bottom]).unique("neighborhood").sort("growth_pct", descending=True)

fig = px.bar(
    chart_data,
    x="growth_pct",
    y="neighborhood",
    orientation="h",
    title="Cumulative Price Growth 2021 to 2025 — Median AED/sqm (Top 30 + Bottom 10)",
    labels={"growth_pct": "Price Growth (%)", "neighborhood": ""},
    color="growth_pct",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    text="growth_pct",
)

fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    height=1100,
    xaxis=dict(ticksuffix="%"),
    coloraxis_showscale=False,
    yaxis=dict(categoryorder="total ascending"),
)
fig.show()

### What the chart tells us

**Golf City and Falcon City of Wonders lead at 224% and 215%.** These are not premium branded addresses. They are mid-market villa communities in the outer fringe of Dubai that started 2021 at very low price per sqm. The enormous percentage growth reflects a low base effect: when you start cheap, any meaningful demand surge produces spectacular percentage returns. These areas did not become the most expensive in Dubai; they simply had the most room to run.

**Jumeirah at 195% is the chart's most significant outlier.** Unlike Golf City or Falcon City, Jumeirah is an established, mature, land-scarce beachside community. It is not a new development area with a low starting base. A near-tripling of median price per sqm in four years in a market that was already expensive and supply-constrained is a genuine scarcity story. Ultra-high-net-worth demand for freehold beachside land drove this, and there is simply no new land to add in Jumeirah.

**Dubai Silicon Oasis and Motor City round out the top five, both exceeding 150%.** These are self-contained master communities that were considered peripheral and undervalued in 2021. The post-pandemic shift toward hybrid work and larger living spaces drove demand for affordable family-sized units outside the traditional core, and both areas benefited enormously from that repricing.

**The 100% club tells the story of the broader cycle.** Emirates Hills, Arabian Ranches, Saih Shuaib, and Al Jaddaf all doubled over the period. These span ultra-luxury (Emirates Hills), established family suburbs (Arabian Ranches), and emerging industrial-adjacent areas (Saih Shuaib, Al Jaddaf). The fact that such diverse areas all doubled confirms this was a market-wide phenomenon, not a story confined to one segment or geography.

**The mature premium core lagged on a percentage basis.** Dubai Marina grew 39.8%, Palm Jumeirah 87.3%, Business Bay 77.8%. These are not weak numbers in absolute terms, but they sit well below the market median on this chart. The reason is simple: they were already expensive in 2021 and had less room to grow on a percentage basis. In absolute AED per sqm terms, they may still have added more value than Golf City; percentage growth just does not capture that.

**DAMAC Hills and DAMAC Hills 2 diverge sharply.** DAMAC Hills grew 39.3% while DAMAC Hills 2 grew only 29.8%. Both are DAMAC master communities but Hills 2 is newer, further out, and still delivering large volumes of new units, which caps price appreciation. Hills, being older and more established with a functioning community and amenities, held its relative value better.

**The bottom of the chart is instructive.** Al Safouh at -2.7% is the only area that declined. Al Qusais Industrial at 6.2% and Um Hurair at 5.3% are effectively flat in real terms. These are not residential investment areas in the traditional sense; Al Qusais Industrial is exactly what the name suggests, and very low transaction volumes in these areas mean the median is sensitive to just a handful of deals. They are statistical footnotes rather than meaningful investment signals.

## Transaction Breakdown: Neighborhood → Property Type → Sale Type

### What are we looking at?
A sunburst chart with three concentric rings. The inner ring shows neighborhoods, the middle ring splits each neighborhood by property type (Flat or Villa), and the outer ring splits further by sale type (Off-Plan or Ready). The size of each segment is proportional to the number of transactions.

### Why this chart?
Every other chart in this notebook looks at one or two dimensions at a time. This chart shows three dimensions simultaneously in a single, interactive view. It lets you drill from the macro (which neighborhoods dominate the market by volume) all the way down to the micro (within a given neighborhood, how much of the villa market is off-plan versus ready) without switching charts.

### What can we learn from it?
- **Which neighborhoods account for the largest share of total market volume** and how concentrated the market is across the top areas
- **Whether a neighborhood's volume is driven by flats or villas** and how that split compares across areas
- **The off-plan vs ready composition at neighborhood level** rather than the market-wide aggregate shown in Chart 2, revealing which specific areas are the engines of Dubai's off-plan boom
- **Structural differences between flat and villa markets within the same neighborhood** — an area might be predominantly ready-market for villas but heavily off-plan for flats, or vice versa

In [ ]:
TOP_N = 20

top_neighborhoods = (
    df.group_by("neighborhood")
      .agg(pl.len().alias("transactions"))
      .sort("transactions", descending=True)
      .head(TOP_N)
      ["neighborhood"].to_list()
)

sunburst_data = (
    df.filter(pl.col("neighborhood").is_in(top_neighborhoods))
      .group_by(["neighborhood", "property_cat", "sale_type"])
      .agg(pl.len().alias("transactions"))
      .sort(["neighborhood", "property_cat", "sale_type"])
)

fig = px.sunburst(
    sunburst_data,
    path=["neighborhood", "property_cat", "sale_type"],
    values="transactions",
    title="Transaction Breakdown by Neighborhood, Property Type & Sale Type (Top 20 Neighborhoods)",
    color="neighborhood",
    color_discrete_sequence=px.colors.qualitative.Pastel,
)

fig.update_layout(height=750)
fig.update_traces(textinfo="label+percent entry")
fig.show()

### What the chart tells us

**The market is highly concentrated.** The top 20 neighborhoods account for the majority of all transactions in the dataset, and within those 20, the top 3 or 4 dominate visually. Jumeirah Village Circle, Business Bay, and Mohammed Bin Rashid City command the largest slices of the inner ring, confirming what volume charts showed earlier: a handful of master communities absorb a disproportionate share of Dubai's residential transaction activity.

**JVC is the volume king and it is almost entirely flats.** The JVC segment (Flat 13%, Off-Plan 10%, Ready 3%) shows the neighborhood is overwhelmingly flat-driven with a roughly 75/25 off-plan to ready split. This matches JVC's identity as Dubai's highest-volume affordable flat market, where developers continue to launch off-plan projects at scale because buyer demand at that price point remains deep.

**Business Bay is a flat-only market.** The Business Bay segment shows essentially no villa presence at all. Every transaction is a flat, and the off-plan share is significant. This is consistent with Business Bay's built environment: a dense, mixed-use urban district with no land for villa development. The entire buyer universe there is either end-user flat buyers or investor units.

**Villanova is the clearest pure-villa outlier in the top 20.** Its segment is divided into Villa and a tiny Flat sliver, with the villa portion dominated by Ready transactions. Villanova is a JLL/Dubai Properties townhouse community delivering completed units, which explains the ready-market skew. It stands out on this chart as the only top-20 neighborhood where villas account for virtually all volume.

**Mohammed Bin Rashid City shows the most balanced Flat/Villa split.** MBR City's segment shows a meaningful villa band alongside a larger flat band, reflecting its nature as a mixed master development containing both Sobha Hartland flats and large standalone villas in the same district boundary. The off-plan share across both property types is high, as MBR City has been one of the most active off-plan launch zones in Dubai across the entire period.

**Dubai South is almost entirely off-plan flats.** The tiny Dubai South slice is almost fully consumed by off-plan, with a negligible ready segment. This is a market still in early-stage delivery. Most of what has been sold there is pre-construction, and the ready market is thin because completions are still catching up to the volume of launches.

**The outer ring confirms the market-wide off-plan dominance.** Looking around the entire outer ring, green (Off-Plan) segments are consistently larger than the ready segments across nearly every neighborhood and property type combination. There are very few areas where ready transactions dominate. The exceptions, such as Dubai Marina's ready flat share and Villanova's ready villa share, are mature or recently delivered communities rather than active launch markets.

---

## Final Thoughts

Seven charts, 566,000 transactions, five years of data. Here is what the Dubai residential market actually looks like when you let the numbers speak.

**The macro story is simple: everything went up, and it did not stop.**
From 2021 to 2025, transaction volume grew roughly 8x and median prices per sqm roughly doubled across the broad market. This was not a bubble followed by a correction. The rolling volume chart shows one of the cleanest sustained uptrends you will find in any major real estate market over this period.

**Off-plan is not a segment of the market. It is the market.**
By 2025, roughly 75-80% of all residential transactions were off-plan. Developers, investors, and the regulatory environment all aligned to produce this outcome. The implications are significant: a large wave of completions is due between 2025 and 2028, and whether the market absorbs it smoothly depends entirely on whether the demand pipeline stays as deep as the supply pipeline.

**The fastest growth was not where you expect it.**
Golf City at 224%, Falcon City at 215%, Dubai Silicon Oasis at 173%. Mid-market and fringe communities outpaced Palm Jumeirah and Dubai Marina on percentage growth by a wide margin. The luxury tier was already expensive in 2021; the real appreciation story of this cycle belongs to the affordable outer belt.

**Jumeirah is the one exception that breaks the rule.**
It is the only established, land-scarce, premium area that also delivered near-200% price growth. No new supply, genuine scarcity, and ultra-high-net-worth demand for freehold beachside land. That combination is rare and the chart reflects it clearly.

**JVC is Dubai's engine, not its showroom.**
The highest transaction volume, a balanced bedroom mix, broad buyer demographic, and sustained off-plan demand. It is not glamorous, but it is the neighborhood that keeps the market functioning at scale. Any analysis of Dubai residential real estate that ignores JVC is missing the point.

**What this dataset cannot tell you** is whether current prices are sustainable. It describes what happened, not what comes next. The pipeline risk is visible in the off-plan chart, the concentration risk is visible in the sunburst, and the scarcity premium is visible in the YoY growth chart. Drawing forward-looking conclusions from these signals requires assumptions about population growth, global capital flows, and developer behaviour that sit outside any historical dataset.

What the data does tell you is that between 2021 and 2025, Dubai was one of the most active and consistently appreciating residential markets on the planet. This project documents that cycle in full.